In [1]:
# 환경 설정
from dotenv import load_dotenv
load_dotenv()

# 라이브러리  
import os
import pandas as pd
from pprint import pprint
import json

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.1
)

In [8]:
# 데이터 로드 (샘플링)
df = pd.read_csv('drug_info.csv')

df_sample = df.head(500)

print(len(df_sample))
df_sample.head(2)

500


,제품명,업체명,품목기준코드,효능효과,사용법,부작용,주의사항,주의사항_경고,상호작용,보관법
0,활명수,동화약품(주),195700020,"이 약은 식욕감퇴(식욕부진), 위부팽만감, 소화불량, 과식, 체함, 구역, 구토에 ...","만 15세 이상 및 성인은 1회 1병(75 mL), 만 11세이상~만 15세미만은 ...",NaN,만 3개월 미만의 젖먹이는 이 약을 복용하지 마십시오.\n\n이 약을 복용하기 전에...,NaN,NaN,습기와 빛을 피해 실온에서 보관하십시오.\n\n어린이의 손이 닿지 않는 곳에 보관하...
1,신신티눈고(살리실산반창고)(수출명:SINSINCORNPLASTER),신신제약(주),195900034,"이 약은 티눈, 못(굳은살), 사마귀에 사용합니다. \n",이형지로부터 벗겨 이 약제면을 환부(질환 부위)에 대고 테이프로 고정하고 2~5일마...,"발진, 발적(충혈되어 붉어짐), 홍반(붉은 반점), 가려움, 정상 피부에 닿았을 경...","이 약에 과민증 환자, 당뇨병, 혈액순환장애 환자는 이 약을 사용하지 마십시오.\n...",NaN,"메토트렉세이트, 설포닐우레아, 다른 국소 적용 약물과 함께 사용 시 의사 또는 약사...",습기와 빛을 피해 보관하십시오.\n\n어린이의 손이 닿지 않는 곳에 보관하십시오.\n


In [11]:
# langfuse
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()


# 연결 확인
from langfuse import Langfuse
langfuse = Langfuse()
assert langfuse.auth_check()
print("Langfuse 연결 확인 완료")

Langfuse 연결 확인 완료


In [13]:
# 키워드 추출 체인
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# langfuse prompt 생성
keyword_prompt = langfuse.create_prompt(
    name="drug-keyword-extractor",
    type="chat",
    prompt=[{
        "role": "system",
        "content": """당신은 의약품 전문가입니다. 주어진 의약품 정보에서 핵심 키워드와 요약을 추출합니다.

        ## 추출 지침:
        - 키워드: 주요 성분, 대상 질환, 효능, 약효 분류, 주의 대상 등 의약품을 대표하는 핵심 키워드 3-5개.
        - 요약: 의약품 정보 전체 요약 1문장.
        """
        },
        {
        "role": "user",
        "content": """{drug_info}"""
        }
    ],
    labels=["production"],
    config={"model": "gpt-4.1-mini", "temperature": 0.1}
)

# 출력 형식 정의
class DrugKeyword(BaseModel):
    keyword: List[str] = Field(description="약품의 가장 핵심 키워드(약효 분류, 성분명, 효능 등)")
    summary: str = Field(description="약품 정보 전체를 1문장으로 요약")

# langfuse prompt 불러오기
langchain_prompt = ChatPromptTemplate.from_messages(keyword_prompt.get_langchain_prompt())
langchain_prompt.metadata = {"langfuse_prompt": keyword_prompt}

# chain 생성
keyword_extractor = langchain_prompt | llm.with_structured_output(DrugKeyword)

In [17]:
test = df_sample.iloc[0]
result = keyword_extractor.invoke(
    {"drug_info": f"제품명: {test['제품명']}\n효능효과: {test['효능효과']}\n부작용: {test['부작용']}"},
    config={"callbacks": [langfuse_handler]}
)
print("키워드:", result.keyword)
print("요약:", result.summary)

키워드: ['식욕감퇴', '소화불량', '위부팽만감', '구역 구토 완화', '소화제']
요약: 활명수는 식욕부진, 위부팽만감, 소화불량, 과식, 체함, 구역 및 구토 증상을 완화하는 소화제입니다.


In [19]:
def create_drug_documents(df):
    all_docs = []

    for idx, row in df.iterrows():
        drug_name = row['제품명']
        print(f"[{idx+1}/{len(df)}] {drug_name} 진행중...")

        input_text = f"제품명: {drug_name}\n"
        if pd.notna(row.get('효능효과')):
            input_text += f"효능효과: {row['효능효과']}\n"
        if pd.notna(row.get('부작용')):
            input_text += f"부작용: {row['부작용']}\n"

        try:
            result = keyword_extractor.invoke(
                {"drug_info": input_text},  # ← drug_info 로 통일
                config={"callbacks": [langfuse_handler]}
            )
            keywords = ", ".join(result.keyword)  # ← List → 문자열
            summary = result.summary
        except Exception as e:
            print(f"  키워드 추출 실패: {e}")
            keywords = drug_name
            summary = f"{drug_name} 관련 의약품 정보"

        for field_col, field_label in FIELDS.items():
            content = row.get(field_col)
            if pd.isna(content):
                continue

            doc = Document(
                page_content=f"{drug_name}의 {field_label}: {content}",
                metadata={
                    'drug_name': drug_name,
                    'company': row['업체명'],
                    'item_code': str(row['품목기준코드']),
                    'field': field_label,
                    'keywords': keywords,
                    'summary': summary,
                    'content': str(content)
                }
            )
            all_docs.append(doc)

    return all_docs

drug_docs = create_drug_documents(df_sample)
print(f"\n생성된 Document 수: {len(drug_docs)}개")
print(drug_docs[0].page_content[:200])
pprint(drug_docs[0].metadata)

[1/500] 활명수 진행중...
[2/500] 신신티눈고(살리실산반창고)(수출명:SINSINCORNPLASTER) 진행중...
[3/500] 아네모정 진행중...
[4/500] 타치온정50밀리그램(글루타티온(환원형)) 진행중...
[5/500] 타치온정50밀리그램(글루타티온(환원형)) 진행중...
[6/500] 겔포스현탁액(인산알루미늄겔) 진행중...
[7/500] 일양노이겔현탁액(규산알루민산마그네슘) 진행중...
[8/500] 자모 진행중...
[9/500] 페니라민정(클로르페니라민말레산염) 진행중...
[10/500] 일양노이시린에이정(규산알루민산마그네슘) 진행중...
[11/500] 세나서트2밀리그람질정 진행중...
[12/500] 대한염화나트륨액 진행중...
[13/500] 옵타젠트점안액(포비돈) 진행중...
[14/500] 삐콤정 진행중...
[15/500] 게루삼정 진행중...
[16/500] 보화소합원(대환,소환) 진행중...
[17/500] 지엘타이밍정(카페인무수물) 진행중...
[18/500] 엔클비액(염화나트륨) 진행중...
[19/500] 지노콜시럽(구연산부타미레이트) 진행중...
[20/500] 제로미아액 진행중...
[21/500] 베스타제정 진행중...
[22/500] 베스타제당의정 진행중...
[23/500] 원비디 진행중...
[24/500] 에어신신파스 진행중...
[25/500] 미보(MEBO)연고 진행중...
[26/500] 한림포비돈점안액(수출명:한비돈점안액) 진행중...
[27/500] 포스테리산좌제 진행중...
[28/500] 판콜에이내복액 진행중...
[29/500] 코푸시럽에스 진행중...
[30/500] 포스테리산연고 진행중...
[31/500] 로와치넥스캡슐 진행중...
[32/500] 아로나민골드정 진행중...
[33/500] 대웅우루사연질캡슐 진행중...
[34/500] 액티피드정 진행중...
[35/500] 태극아즈렌에스연고(구아야줄렌) 진행중...
[36/500] 보나링에이정(디멘히드리네이트) 진행중...

In [26]:
import json

with open("./drug_docs.json", 'w', encoding='utf-8-sig') as f:
    json.dump([doc.model_dump() for doc in drug_docs], f, indent=2, ensure_ascii=False)

print("저장 완료!")

저장 완료!


In [3]:
from langchain_core.documents import Document
import json

with open("./drug_docs.json", 'r', encoding='utf-8-sig') as f:
    drug_docs = [Document(**doc) for doc in json.load(f)]

print(f"로드된 Document 수: {len(drug_docs)}개")

로드된 Document 수: 2723개


In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# chroma 생성
vector_store = Chroma.from_documents(
    documents=drug_docs,
    embedding=embeddings,
    collection_name="drug_info_db",
    persist_directory="./chroma_db",
)

print(f"저장된 Document 수: {vector_store._collection.count()}개")

저장된 Document 수: 2723개


In [5]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)

# 테스트
query = "타이레놀 먹고 속이 아파요"
results = mmr_retriever.invoke(query)

for result in results:
    print(f"약품: {result.metadata['drug_name']}")
    print(f"필드: {result.metadata['field']}")
    print(f"키워드: {result.metadata['keywords']}")
    print(f"내용: {result.page_content[:100]}...")
    print("=" * 50)

약품: 타스멘정(아세트아미노펜)
필드: 보관법
키워드: 아세트아미노펜, 감기 발열 및 통증 완화, 진통제 및 해열제, 부작용: 알레르기 반응, 간기능 이상, 주의: 혈액 이상 및 위장 출혈 가능성
내용: 타스멘정(아세트아미노펜)의 보관법: 실온에서 보관하십시오.

어린이의 손이 닿지 않는 곳에 보관하십시오.
...
약품: 알게이트정(알마게이트)
필드: 효능효과
키워드: 알마게이트, 위궤양, 제산작용, 위염, 부작용: 변비, 설사
내용: 알게이트정(알마게이트)의 효능효과: 이 약은 위·십이지장궤양, 위염, 위산과다, 속쓰림, 구역, 구토, 위통, 신트림의 제산작용 및 증상의 개선에 사용합니다.
...
약품: 옵타젠트점안액(포비돈)
필드: 부작용
키워드: 포비돈, 건조한 눈, 하드콘택트렌즈 착용, 점안액, 과민반응 주의
내용: 옵타젠트점안액(포비돈)의 부작용: 매우 드물게 과민반응이 나타날 수 있습니다....
약품: 아비나파스타(트리암시놀론아세토니드)
필드: 보관법
키워드: 트리암시놀론아세토니드, 만성 박리성 치은염, 난치성 구내염 및 설염, 염증 완화, 구강 내 부작용 주의
내용: 아비나파스타(트리암시놀론아세토니드)의 보관법: 실온에서 보관하십시오....
